# Kernel

Saved runs of one topic read as kernels - a matrix with one row per outcome and one column per distinct profile - and analysed block by block, after `CoRe/llm_pushforward`. A grid is a designed kernel (every combination of its factors' levels); a survey run is a sampled one. The first cell is the only setup: the topic and the runs (each by any text only its stem contains; a mapping label -> text names them), the name the written files carry, and whether anything is written. `FACTORS` names the profile's columns (a list); `None` takes the columns each run's description reads. `REFERENCE` is the label the agreement and the paired intervals hold the others against; `None` takes the model trained on the subjects' answers where there is one. `DRAWS` and `SHAPLEY_DRAWS` are the replicates of the intervals. `python kernel.py <topic> <run> ...` runs the same blocks in one go.

In [ ]:
%matplotlib inline
from panel import paths
from panel.analysis import Output
from panel.analysis.comparison import load_runs
from panel.analysis.kernel import (
    agreement, axes, describe, electorates, importance, intervals, load_kernels, noise, pairs, populations, scales,
    sensitivity, shapley, surrogate, variance,
)
from panel.run import ResultStore

TOPIC = "election_2021"
RUN_NAMES = {
    "gpt-4o": "sod_legacy_cslegacy_legacy_OpenAI_gpt-4o_Prob", "gpt-4.1": "sod_legacy_cslegacy_legacy_OpenAI_gpt-4.1_Prob",
    "gpt-4o-mini": "sod_legacy_cslegacy_legacy_OpenAI_gpt-4o-mini_Prob", "grid sod": "gridsod_",
}
NAME = TOPIC
SAVE = False
FACTORS = None
REFERENCE = "gpt-4o"
DRAWS = 200
SHAPLEY_DRAWS = 20
STYLED = True

STORE = ResultStore(paths.RESULTS_DIR, TOPIC)
KERNELS = load_kernels(load_runs(STORE, RUN_NAMES, styled=STYLED), FACTORS)
OUT = Output(STORE, NAME, save=SAVE)
print(describe(KERNELS))

## Noise floor

Two answers to the same profile - repeats of one subject, or subjects sharing every value - against two random subjects. Every distance below is read against it.

In [ ]:
table, figures = noise(KERNELS, OUT)
display(table.round(3), *figures)

## Scales

The TV between every two profiles' answers: the Dobrushin coefficient δ (the largest, which bounds how far a population's answer can move) and its robust stand-in, the 99th percentile; how far a profile is from the mean answer; how much each outcome moves over the profiles.

In [ ]:
table, figures = scales(KERNELS, OUT)
display(table.round(3), *figures)

## Minimal pairs

Two profiles differing in one factor only: what swapping that value does to the answer, against the noise floor. A grid pairs every profile with every other level of every factor.

In [ ]:
table, figures = pairs(KERNELS, OUT)
display(table.round(3), *figures)

## What the variation is made of

Main effects, two-way interactions, noise and the rest of each outcome's variance over the profiles: exact for a grid, out-of-sample ridge R² for a survey run.

In [ ]:
table, figures = variance(KERNELS, OUT)
display(table.round(3), *figures)

## What each factor explains alone

ε², the bias-corrected share of the variance a factor's level means account for, pooled and per outcome, and the largest TV between two of its levels' mean answers.

In [ ]:
table, figures = importance(KERNELS, OUT)
display(table.round(3), *figures)

## Axes

The SVD of each kernel with each outcome's mean taken off: how many directions the profiles move the answer in, what each direction does to the outcomes, which factor drives it, and the kernels as images sorted along the first.

In [ ]:
table, figures = axes(KERNELS, OUT)
display(table.round(3), *figures)

## Shapley effects

The variance all factors explain together, shared fairly among them; columns the dataset declares `together` (region and district) are one player. Also for the scores on the first two axes. Intervals from the kernel's replicates.

In [ ]:
table, figures = shapley(KERNELS, OUT, draws=SHAPLEY_DRAWS)
display(table.round(3), *figures)

## Electorates

How alike the options' electorates are, factor by factor.

In [ ]:
table, figures = electorates(KERNELS, OUT)
display(table.round(3), *figures)

## Populations

What the run's own population chooses by the kernel, what every profile counted once chooses, what the subjects claimed, and what evening out one factor's make-up does.

In [ ]:
table, figures = populations(KERNELS, OUT)
display(table.round(3), *figures)

## Sensitivity

The largest gain and loss of each outcome when a tenth of the population moves anywhere, and when only one factor's distribution moves.

In [ ]:
table, figures = sensitivity(KERNELS, OUT)
display(table.round(3), *figures)

## Additive surrogate

The kernel without interactions: how additive it is, the largest TV and each outcome's range over every combination of levels, and the population with independent factors.

In [ ]:
table, figures = surrogate(KERNELS, OUT)
display(table, *figures)

## Agreement

Every kernel against the reference, profile by profile on the profiles both have: TV, correlation per outcome, and the ratio of spreads.

In [ ]:
table, figures = agreement(KERNELS, OUT, REFERENCE)
display(table.round(3), *figures)

## Intervals

95 % intervals for the main statistics: half-samples of a survey run's profiles, resampled repeats of a grid with repeats; paired differences from `REFERENCE` where two kernels share their profiles.

In [ ]:
table, figures = intervals(KERNELS, OUT, draws=DRAWS, reference=REFERENCE)
display(table.round(3), *figures)